In [31]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [32]:
# TotalCharges sometimes has blanks -> convert to numeric, drop broken rows
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

print("After TotalCharges cleaning:", df.shape)


After TotalCharges cleaning: (7032, 21)


In [33]:
# Make sure Churn is clean text, then encode
df["Churn"] = df["Churn"].astype(str).str.strip()
df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0})

print("NaNs in Churn:", df["Churn"].isna().sum())
df["Churn"].value_counts()


NaNs in Churn: 0


/tmp/ipython-input-2948593847.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Churn"] = df["Churn"].replace({"Yes": 1, "No": 0})


,count
Churn,
0,5163
1,1869


In [34]:
# Change categorical columns into numeric columns
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

print("X:", X.shape, "| y:", y.shape)


X: (7032, 7061) | y: (7032,)


In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)


Train: (5625, 7061) | Test: (1407, 7061)


In [36]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print("Random Forest is trained")


Random Forest is trained


In [37]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7945984363894811
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1033
           1       0.66      0.47      0.55       374

    accuracy                           0.79      1407
   macro avg       0.74      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [38]:
import numpy as np

importances = rf.feature_importances_
top_idx = np.argsort(importances)[-10:][::-1]

top_features = X.columns[top_idx]
top_importances = importances[top_idx]

list(zip(top_features, top_importances))


[('TotalCharges', np.float64(0.1019463291261027)),
 ('tenure', np.float64(0.09537685556443527)),
 ('MonthlyCharges', np.float64(0.0814502318886576)),
 ('InternetService_Fiber optic', np.float64(0.028537966132165755)),
 ('PaymentMethod_Electronic check', np.float64(0.024252427290643985)),
 ('Contract_Two year', np.float64(0.024187423290293653)),
 ('OnlineSecurity_Yes', np.float64(0.020159508299499633)),
 ('TechSupport_Yes', np.float64(0.019356906375404746)),
 ('Contract_One year', np.float64(0.018015389249857888)),
 ('PaperlessBilling_Yes', np.float64(0.017450816325245384))]

In [39]:
import joblib

joblib.dump(rf, "churn_model.joblib")
joblib.dump(X.columns.tolist(), "model_columns.joblib")

print("Saved files for Streamlit")
print("churn_model.joblib")
print("model_columns.joblib")


Saved files for Streamlit
churn_model.joblib
model_columns.joblib


In [40]:
from google.colab import files

files.download("churn_model.joblib")
files.download("model_columns.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>